# llmpic — Natural Language Chart Generation SDK

> Describe what you want. Get a chart. 11 types. PNG/SVG/PDF. Inline Jupyter display.

## 1. Initialize SDK

In [ ]:
import os
from llmpic import llmPIC

# Read API key from environment, or paste directly
lp = llmPIC(
    api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
    base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
    model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    max_tokens=4096,
)
print(f"SDK ready — model: {lp.model}")

## 2. Basic — One line, one chart

In [ ]:
r = lp.plot("Monthly sales trend over the past 12 months").render()

if r.success:
    r.show()  # ← Renders inline in Jupyter
    print(f"tokens: in={r.token_usage['input']}, out={r.token_usage['output']}")
else:
    print(f"Failed: {r.error_message}")

## 3. With Data & Style

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.DataFrame({
    "Month": ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"],
    "Revenue": np.random.randint(80, 200, 12),
    "Profit": np.random.randint(15, 50, 12),
})

r = lp.plot("Monthly revenue and profit trends").data(df).style({
    "figsize": [14, 6],
    "color_scheme": "cool",
}).render()

r.show()

## 4. All Chart Types

In [ ]:
from IPython.display import display, Markdown

charts = [
    ("line",     lp.plot("Monthly revenue trend for 2024")),
    ("scatter",  lp.scatter("Random scatter plot with 50 points")),
    ("bar",      lp.bar("Department budget: R&D=200, Marketing=150, Sales=180, HR=100")),
    ("pie",      lp.pie("Market share: A=40%, B=25%, C=20%, Other=15%")),
    ("hist",     lp.hist("Normal distribution, mean 0 std 1, 1000 samples")),
    ("heatmap",  lp.heatmap("5x5 correlation matrix")),
    ("boxplot",  lp.boxplot("Distribution comparison across A/B/C/D groups")),
    ("area",     lp.area("Revenue composition by product line 2020-2024")),
    ("radar",    lp.radar("Product score: Performance=4, Usability=3, Reliability=5, Price=2, Support=4")),
    ("subplots", lp.subplots("2x2 dashboard: line, bar, scatter, pie")),
]

for name, builder in charts:
    r = builder.render()
    if r.success:
        display(Markdown(f"### {name}"))
        r.show()
    else:
        display(Markdown(f"### {name} — {r.error_message[:100]}"))

## 5. Iterative Editing — Refine with natural language

In [ ]:
from IPython.display import display, Markdown

v1 = lp.plot("Quarterly sales: Q1=100, Q2=150, Q3=120, Q4=180").render()
display(Markdown("### v1 — Initial"))
v1.show()

v2 = v1.edit("Change to bar chart, use blue color scheme")
display(Markdown("### v2 — Bar chart + blue"))
v2.show()

v3 = v2.edit("Title '2025 Annual Sales Report', increase title size, add grid")
display(Markdown("### v3 — Refined title + style"))
v3.show()

## 6. Multi-Format Export & Save

In [ ]:
r = lp.plot("sin(x) from 0 to 2pi").render()

# One save() for all formats — extension auto-detects
r.save("demo_chart.png")   # PNG
r.save("demo_chart.svg")   # SVG vector
r.save("demo_chart.pdf")   # PDF

# No path → defaults to ~/llmpic_charts/chart_{timestamp}.png
r.save()

# base64 for web embedding
print(f"PNG base64 length: {len(r.base64())} chars")
print(f"SVG base64 length: {len(r.base64_svg())} chars")

## 7. Async Batch Generation

In [ ]:
from llmpic import AsyncllmPIC
from IPython.display import display, Markdown
import time

async def run_batch():
    lp_async = AsyncllmPIC(
        api_key=os.environ.get("LLMPIC_API_KEY", "sk-your-key"),
        base_url=os.environ.get("LLMPIC_BASE_URL", "https://api.openai.com/v1"),
        model=os.environ.get("LLMPIC_MODEL", "gpt-4o"),
    )
    
    t0 = time.time()
    results = await lp_async.batch([
        ("plot", "National 12-month sales trend"),
        ("bar", "Sales comparison by region"),
        ("pie", "Market share distribution"),
        ("scatter", "Customer age vs spend"),
    ])
    
    for i, r in enumerate(results):
        if r.success:
            display(Markdown(f"### batch[{i}]"))
            r.show()
        else:
            print(f"[{i}] Failed: {r.error_message[:100]}")
    
    print(f"4 charts concurrently, total {time.time()-t0:.1f}s")

await run_batch()